# 03 · Fraud EDA (IEEE-CIS) – why the split must be time-based

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parents[1] if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
RAW = ROOT / "data/raw/home-credit"

FRAUD = ROOT / "data/raw/ieee-fraud"

In [ ]:
tx = pd.read_csv(FRAUD / "train_transaction.csv", usecols=["TransactionID", "isFraud", "TransactionDT", "TransactionAmt", "ProductCD", "card1", "addr1", "D1"])
tx.isFraud.mean(), tx.TransactionDT.max() / 86400

In [ ]:
day = tx.TransactionDT // 86400
tx.isFraud.groupby(day // 7).mean().plot(title="weekly fraud rate"); plt.show()

Same cards appear across the whole period: a random split puts a card's future in training → optimistic AUC. We train on the first 70%, validate on the next 10%, test on the last 20% of time.

In [ ]:
cards_per_week = tx.groupby(day // 7).card1.nunique()
shared = len(set(tx.card1[day < 120]) & set(tx.card1[day >= 150])) / tx.card1[day >= 150].nunique()
print(f"{shared:.0%} of test-period cards were already seen in training")